# DS-A1 — Data Provenance and Measurement Audit

**Dataset:** NASA/GSFC SPDF OMNI 2 daily averages (`omni_01_av.dat`), frozen local copy `data/omni_01_av.txt`.

**Testable question:** Within the available daily records, is a more southward daily mean interplanetary magnetic field component (GSM Bz ≤ −5 nT) associated with a higher daily Kp than days with Bz > −5 nT? This is an association question, not a causal or forecasting claim.

**Frozen SHA-256:** `1dd1327e601713be59903b064dbed45421673040f443f4148fd11a4deb4e0c1c`

The official OMNI documentation says daily averages are arithmetic averages of the basic hourly values; no minimum number of contributing points is required. Kp is specially averaged and rounded to standard Kp values.


In [ ]:
from pathlib import Path
import hashlib, pandas as pd, numpy as np
import matplotlib.pyplot as plt
DATA=Path('data/omni_01_av.txt')
EXPECTED_SHA256='1dd1327e601713be59903b064dbed45421673040f443f4148fd11a4deb4e0c1c'
print('SHA256:',hashlib.sha256(DATA.read_bytes()).hexdigest())
assert hashlib.sha256(DATA.read_bytes()).hexdigest()==EXPECTED_SHA256


In [ ]:
cols=['year', 'doy', 'hour', 'bartels', 'imf_sc', 'plasma_sc', 'n_imf', 'n_plasma', 'B_mag', 'B_vec_mag', 'B_lat', 'B_lon', 'Bx_GSE', 'By_GSE', 'Bz_GSE', 'By_GSM', 'Bz_GSM', 'sigma_Bmag', 'sigma_B', 'sigma_Bx', 'sigma_By', 'sigma_Bz', 'proton_temp', 'proton_density', 'flow_speed', 'flow_lon', 'flow_lat', 'Na_Np', 'flow_pressure', 'sigma_T', 'sigma_N', 'sigma_V', 'sigma_phi_V', 'sigma_theta_V', 'sigma_Na_Np', 'electric_field', 'plasma_beta', 'alfven_mach', 'Kp', 'sunspot_R', 'Dst', 'AE', 'proton_gt1', 'proton_gt2', 'proton_gt4', 'proton_gt10', 'proton_gt30', 'proton_gt60', 'proton_flag', 'ap', 'F10_7', 'PCN', 'AL', 'AU', 'magnetosonic_mach']
df=pd.read_csv(DATA,sep=r'\s+',header=None,names=cols,engine='python')
print('shape=',df.shape)
assert df.shape[1]==55
df.head()


## Provenance and data-generating process

Source observations come from multiple near-Earth spacecraft and are cross-normalized/combined into OMNI. Hourly OMNI2 records contain magnetic-field, solar-wind, geomagnetic, solar-activity and energetic-particle variables. The daily file is derived by arithmetic averaging of hourly OMNI2 values. The daily record retains special fields such as counts of contributing hourly points.

**Lineage:** spacecraft/instruments → source measurements → OMNI cross-normalization/time shifting → hourly OMNI2 → arithmetic daily aggregation → frozen local copy → parsing and audit → association analysis.

**Population:** all calendar days represented by this frozen OMNI daily file (1963-01-01 through 2026-12-31 record slots; later 2026 slots are fill-valued). **Sample:** 23,376 daily rows in the frozen file. **Unit:** one daily record/day. **Stakeholder:** a space-weather data user assessing whether the file is fit for an exploratory geomagnetic-association question. **Target:** daily Kp. **Exposure:** daily mean Bz GSM. **Estimand:** difference in mean daily Kp (standard Kp units) between Bz ≤ −5 nT and Bz > −5 nT among days observed for both variables.


In [ ]:
# Schema, duplicates, calendar checks
print('columns:',len(df.columns))
print('duplicate rows:',df.duplicated().sum())
print('duplicate dates:',df.duplicated(['year','doy']).sum())
dates=pd.to_datetime(df.year.astype(str)+df.doy.astype(str),format='%Y%j',errors='coerce')
print('invalid calendar dates:',dates.isna().sum())
assert len(df.columns)==55
assert df.duplicated(['year','doy']).sum()==0
assert dates.isna().sum()==0


In [ ]:
# Missingness: first the deliberately inadequate NaN-only check, then documented sentinel values
print('NaN cells:',int(df.isna().sum().sum()))
fill={"Bz_GSM":999.9,"B_mag":999.9,"proton_density":999.9,"flow_speed":9999,"Kp":99,"Dst":99999,"F10_7":999.9,"ap":999,"sunspot_R":999}
for c,f in fill.items():
    print(c, 'sentinel missing:', int((df[c]==f).sum()), f'({(df[c]==f).mean()*100:.2f}%)')


In [ ]:
# Range/anomaly checks using documented field semantics
checks={
'Bz outside +/-100 nT':((df.Bz_GSM!=999.9)&((df.Bz_GSM<-100)|(df.Bz_GSM>100))).sum(),
'flow speed outside 0-2000 km/s':((df.flow_speed!=9999)&((df.flow_speed<0)|(df.flow_speed>2000))).sum(),
'negative proton density':((df.proton_density!=999.9)&(df.proton_density<0)).sum(),
'invalid Kp code':((df.Kp!=99)&((df.Kp<0)|(df.Kp>90))).sum(),
'negative F10.7':((df.F10_7!=999.9)&(df.F10_7<0)).sum()}
checks


In [ ]:
# Leakage boundary and analysis
a=df[(df.Bz_GSM!=999.9)&(df.Kp!=99)].copy()
a['Kp_standard']=a.Kp/10
a['group']=np.where(a.Bz_GSM<=-5,'Bz <= -5 nT','Bz > -5 nT')
summary=a.groupby('group').Kp_standard.agg(['count','mean','median','std'])
print(summary)
print('mean difference:',summary.loc['Bz <= -5 nT','mean']-summary.loc['Bz > -5 nT','mean'])
print('Pearson r:',a[['Bz_GSM','Kp_standard']].corr().iloc[0,1])


In [ ]:
plt.figure(figsize=(8,4.8)); plt.scatter(a.Bz_GSM,a.Kp_standard,s=5,alpha=.25); plt.axvline(-5,linestyle='--'); plt.xlabel('Daily mean Bz (GSM), nT'); plt.ylabel('Daily Kp (standard units)'); plt.title('Daily mean Bz vs daily Kp, 1963–2026'); plt.tight_layout(); plt.show()


## Counterexample, prohibited claim, and limits

**Measurement counterexample:** a day with a short, intense southward Bz interval can have a daily mean Bz above −5 nT after averaging with quieter hours, so a threshold on the daily mean can classify that day as unexposed even though an important sub-daily event occurred. Conversely, daily Kp is a derived/rounded index rather than a continuous physical measurement.

**Prohibited claim:** do not conclude that southward Bz *causes* higher Kp from this analysis. The same-day association does not establish causality, and the daily aggregation obscures timing. Also do not generalize to local geomagnetic conditions at a particular station, to sub-daily storm intensity, or to a future/predictive model.

**Leakage boundary:** for this descriptive association, both variables are allowed because the estimand is same-day association. They must not be used as a leakage-free forecasting setup without defining a prediction horizon; using same-day Kp to predict same-day Kp would be leakage.
